# Photoreal zones on a free GPU (Colab or Kaggle T4)

Runs the repo's photoreal route (`scripts/photoreal_twin.py`) with the trainer on CUDA, at full 960 px frames and more steps than the CPU box can afford. Costs 0 World Labs credits.

1. Runtime > Change runtime type > T4 GPU.
2. Upload `Inside-the-NEW-SnapYT.App.mp4` when asked (or mount Drive and set `VIDEO`).
3. Run all. Each zone takes roughly 10 to 25 minutes. Download `photoreal_zones.zip` at the end and commit its `public/zones/` files, or hand it to Claude in the project.

In [ ]:
!git clone -q -b claude/photoreal-twin-mt4xxb https://github.com/pikulsomesh/foodbank-spatial-twin.git
%cd foodbank-spatial-twin
!apt-get -qq install -y colmap > /dev/null
!pip -q install mediapipe "usd-convert-gsplat[usd]>=0.1.15" scipy opencv-python-headless
import torch; print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import os
VIDEO = "/content/Inside-the-NEW-SnapYT.App.mp4"
if not os.path.exists(VIDEO):
    from google.colab import files
    up = files.upload()
    VIDEO = "/content/" + next(iter(up))
ZONES = ["racking", "market", "sorting"]
ITERS = 15000

In [ ]:
import subprocess
def run(*a):
    print("$", " ".join(a)); subprocess.run(a, check=True)
for z in ZONES:
    run("python", "scripts/photoreal_twin.py", "frames", "--zone", z, "--video", VIDEO)
    run("python", "scripts/photoreal_twin.py", "masks", "--zone", z)
    run("python", "scripts/photoreal_twin.py", "select", "--zone", z)
    extra = ["--matcher", "exhaustive", "--init-tri-angle", "3"] if z == "sorting" else []
    run("python", "scripts/photoreal_twin.py", "pose", "--zone", z, *extra)
    run("python", "scripts/photoreal_twin.py", "train", "--zone", z, "--iters", str(ITERS),
        "--train-args", "--width", "960", "--max-splats", "1500000")
    run("python", "scripts/photoreal_twin.py", "export", "--zone", z)

In [ ]:
from IPython.display import Image, display
import glob
for z in ZONES:
    p = sorted(glob.glob(f"build/photoreal/{z}/splat/preview_*.jpg"))
    if p: print(z); display(Image(p[-1], width=900))

In [ ]:
!zip -qr photoreal_zones.zip public/zones/photoreal.json public/zones/*-3dgs.spz public/zones/*-3dgs.usdz build/photoreal/*/splat/preview_*.jpg
from google.colab import files; files.download("photoreal_zones.zip")